# NDBC buoys (tabledap -> DataFrame)

A `tabledap` dataset returns record tables as a `pandas.DataFrame`. Here we pull a day of NDBC standard-meteorological buoy water-temperature observations off the U.S. west coast.

## Setup + request

`OUTPUT_KIND` is `tabular`, so `download()` returns a frame (also written to CSV). `aggregate=` would be rejected for a table.

In [1]:
import tempfile
from pathlib import Path

from earthlens import EarthLens

out_dir = Path(tempfile.mkdtemp(prefix='earthlens-erddap-ndbc-'))
df = EarthLens(
    data_source='erddap',
    dataset='cwwcNDBCMet',
    variables=['station', 'time', 'wtmp'],
    start='2023-01-01',
    end='2023-01-01',
    lat_lim=[36.0, 38.0],
    lon_lim=[-124.0, -122.0],
    path=out_dir,
).download()

print('shape  :', df.shape)
print('columns:', list(df.columns))

2026-06-25 19:51:13.301 | INFO     | earthlens.erddap.backend:download:491 - ERDDAP cwwcNDBCMet: 17 row(s) written to C:\Users\main\AppData\Local\Temp\earthlens-erddap-ndbc-0pk_is6v\cwwcNDBCMet.csv


shape  : (17, 3)
columns: ['station', 'time (UTC)', 'wtmp (degree_C)']


## Peek at the records

ERDDAP appends units to the column names (e.g. `wtmp (degree_C)`).

In [2]:
df.head()

,station,time (UTC),wtmp (degree_C)
0,46012,2023-01-01T00:00:00Z,NaN
1,46026,2023-01-01T00:00:00Z,12.3
2,46042,2023-01-01T00:00:00Z,NaN
3,46237,2023-01-01T00:00:00Z,11.7
4,46269,2023-01-01T00:00:00Z,12.5


## Per-station water temperature

In [3]:
wtmp_col = [c for c in df.columns if c.startswith('wtmp')][0]
summary = (
    df.dropna(subset=[wtmp_col])
    .groupby('station')[wtmp_col]
    .agg(['count', 'mean', 'min', 'max'])
    .round(2)
)
print(f'{len(summary)} station(s) reported water temperature:')
summary

8 station(s) reported water temperature:


,count,mean,min,max
station,,,,
46026,1,12.3,12.3,12.3
46237,1,11.7,11.7,11.7
46269,1,12.5,12.5,12.5
AAMC1,1,11.4,11.4,11.4
FTPC1,1,12.8,12.8,12.8
PRYC1,1,11.9,11.9,11.9
RCMC1,1,11.9,11.9,11.9
RTYC1,1,13.3,13.3,13.3
